# Is it useful ?!

In [1]:
import os
import pandas as pd

DATA_DIR = "/nfs/scratch/pdb_dimers/KaHIP"

In [14]:
output = pd.read_csv(os.path.join(DATA_DIR, "output.txt"), header=None)
output.head()

,0
0,3
1,3
2,3
3,3
4,3


In [15]:
output["cluster_id"] = output.index.map(lambda x: x + 1)  # METIS uses 1-based indexing
output.head()

,0,cluster_id
0,3,1
1,3,2
2,3,3
3,3,4
4,3,5


In [16]:
# Plot the distribution of cluster sizes
cluster_sizes = output[0].value_counts().sort_index()
print(cluster_sizes)

0
0    799
1    799
2    798
3    796
4    798
5    799
6    798
7    798
8    799
9    799
Name: count, dtype: int64


# Partitioning with sequence identity

In [2]:
seq_ident_output = pd.read_csv(os.path.join(DATA_DIR, "seq_ident_partitions_strong_output.txt"), header=None)
seq_ident_output.head()

,0
0,8
1,7
2,8
3,2
4,2


In [5]:
len(seq_ident_output)

22093

In [4]:
seq_ident_cluster_sizes = seq_ident_output[0].value_counts().sort_index()
print(seq_ident_cluster_sizes)

0
0    2275
1    2275
2    2275
3    2275
4    2275
5    2275
6    2275
7    2275
8    1618
9    2275
Name: count, dtype: int64


# Fuse partition plus into interaction df

In [16]:
int_df = pd.read_csv("/nfs/scratch/pdb_dimers/final_filtered_interactions.tsv", sep="\t")
partitions = pd.read_csv("/nfs/scratch/pdb_dimers/KaHIP/seq_ident_partitions_strong_output.txt", header=None)

partitions.head()

,0
0,8
1,7
2,8
3,2
4,2


In [17]:
len(partitions)

22093

In [18]:
partitions = partitions[0]

int_df["partitions"] = partitions

int_df.head()

,assembly_id,pdb_id,assembly_number,entity_pair,uniprot_pair,uniprot_1,uniprot_2,species_pair,species_1,species_2,...,dimer_type,cluster_pair_100pct,new_cluster_pair,resolution_best_angstrom,modeled_polymer_monomer_count,experimental_method,oligomeric_count,download_url,local_filename,partitions
0,10BL-1,10BL,1,"10BL_1,10BL_1",Q4E2L0|Q4E2L0,Q4E2L0,Q4E2L0,Trypanosoma cruzi|Trypanosoma cruzi,Trypanosoma cruzi,Trypanosoma cruzi,...,homo,"20385_100,20385_100","fix_18845_100,fix_18845_100",2.60,664,X-ray,2,https://files.rcsb.org/download/10bl-assembly1...,10bl-assembly1.cif.gz,8
1,10FT-1,10FT,1,"10FT_1,10FT_2",Q99435|Q78DX7,Q99435,Q78DX7,Homo sapiens|Mus musculus,Homo sapiens,Mus musculus,...,hetero,"10609_100,17054_100","fix_14530_100,fix_37352_100",3.21,775,EM,2,https://files.rcsb.org/download/10ft-assembly1...,10ft-assembly1.cif.gz,7
2,10JU-1,10JU,1,"10JU_1,10JU_2",Q582V7|Q582V7,Q582V7,Q582V7,Trypanosoma brucei brucei TREU927|Trypanosoma ...,Trypanosoma brucei brucei TREU927,Trypanosoma brucei brucei TREU927,...,hetero,"65494_100,65494_100","fix_18848_100,fix_18846_100",2.15,522,X-ray,2,https://files.rcsb.org/download/10ju-assembly1...,10ju-assembly1.cif.gz,8
3,10LI-1,10LI,1,"10LI_1,10LI_1",A0A0D6FAR3|A0A0D6FAR3,A0A0D6FAR3,A0A0D6FAR3,Salmonella enterica subsp. enterica serovar Ty...,Salmonella enterica subsp. enterica serovar Ty...,Salmonella enterica subsp. enterica serovar Ty...,...,homo,"50102_100,50102_100","fix_31296_100,fix_31296_100",1.67,924,X-ray,2,https://files.rcsb.org/download/10li-assembly1...,10li-assembly1.cif.gz,2
4,10NM-1,10NM,1,"10NM_1,10NM_1",K4FZF8|K4FZF8,K4FZF8,K4FZF8,Glycine max|Glycine max,Glycine max,Glycine max,...,homo,"41941_100,41941_100","fix_20451_100,fix_20451_100",2.92,897,EM,2,https://files.rcsb.org/download/10nm-assembly1...,10nm-assembly1.cif.gz,2


In [20]:
int_df["split"] = int_df["partitions"].map({
    0: "val",
    1: "train",
    2: "train",
    3: "train",
    4: "train",
    5: "train",
    6: "train",
    7: "test",
    8: "train",
    9: "train",
})

In [21]:
int_df.to_csv("/nfs/scratch/pdb_dimers/final_filtered_interactions_with_partitions.tsv", sep="\t")

# homo vs hetero distribution in different splits

In [22]:
int_df = pd.read_csv("/nfs/scratch/pdb_dimers/final_filtered_interactions_with_partitions.tsv", sep="\t")


int_df_train = int_df[int_df["split"]=="train"]
int_df_val = int_df[int_df["split"]=="val"]
int_df_test = int_df[int_df["split"]=="test"]

len(int_df_test)

2275

In [23]:
def calculate_percentages(df):
    return df["dimer_type"].value_counts(normalize=True) * 100


calculate_percentages(int_df_train)

dimer_type
homo      78.874765
hetero    21.125235
Name: proportion, dtype: float64

In [24]:
calculate_percentages(int_df_val)

dimer_type
homo      83.648352
hetero    16.351648
Name: proportion, dtype: float64

In [25]:
calculate_percentages(int_df_test)

dimer_type
homo      82.065934
hetero    17.934066
Name: proportion, dtype: float64

In [26]:
all_homo = 0
all_hetero = 0
for i in range(10):
    int_df_tmp = int_df[int_df["partitions"] == i]
    counts = int_df_tmp["dimer_type"].value_counts(normalize=True)*100
    all_homo += counts["homo"]
    all_hetero += counts["hetero"]
    print(f"{i}:\thomo: {counts["homo"]}\thetero: {counts["hetero"]}")

print("\n")
print(f"Avg Counts:\nhomo: {all_homo/10}\thetero: {all_hetero/10}")

0:	homo: 83.64835164835165	hetero: 16.35164835164835
1:	homo: 88.17582417582418	hetero: 11.824175824175825
2:	homo: 90.15384615384615	hetero: 9.846153846153847
3:	homo: 86.5934065934066	hetero: 13.406593406593407
4:	homo: 89.71428571428571	hetero: 10.285714285714285
5:	homo: 35.73626373626374	hetero: 64.26373626373626
6:	homo: 66.15384615384615	hetero: 33.84615384615385
7:	homo: 82.06593406593406	hetero: 17.934065934065934
8:	homo: 85.66131025957972	hetero: 14.338689740420271
9:	homo: 90.76923076923077	hetero: 9.230769230769232


Avg Counts:
homo: 79.86722992705688	hetero: 20.132770072943124
